# Traffic Pattern Analysis — Day-1 Solo Work

Honest, leakage-safe hourly traffic forecasting built on **day one**, before the team switched plans. This notebook only *displays and explains* the results; all computation lives in `src/*.py` and `viz/*.py` and is reproduced by `python run_pipeline.py`.

## The three-step method
1. **LightGBM** forecasts the **seen** series (stations with history): a per-series weekly climatology carries the shape, LightGBM learns a correction from calendar + roadway features, and split-conformal residuals give the 90% intervals.
2. **A graph neural network (GCN)** forecasts the **cold-start** series (stations with *no* history): message passing over the road network (`network_edges.csv`) interpolates each station's traffic *level* from its neighbours, blended with a peer-group median prior; the hourly *shape* comes from its peer group — `functional_class` where present, and the `aadt_band` rung for the 28% of stations where that field is blank.
3. **Combine by routing**: each of the 170,956 target rows is served by the model that applies to it — history → LightGBM, no history → GNN — into one validated submission.

## Headline figures

The table below is **read from `outputs/metrics.json`** rather than typed in, so it cannot drift away from the code. Re-run `python run_pipeline.py` and it updates itself.

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

M = json.load(open('outputs/metrics.json'))

headline = pd.DataFrame(
    [
        ('MAE (veh/hr)',        f"{M['MAE']:.0f}",              '—'),
        ('RMSE',                f"{M['RMSE']:.0f}",             '—'),
        ('R2',                  f"{M['R2']:.3f}",               '—'),
        ('WAPE',                f"{M['WAPE']:.3f}",             f"{M['cold_sim_WAPE']:.3f}"),
        ('WAPE, naive baseline',f"{M['wape_naive']:.3f}",       f"{M['cold_naive_WAPE']:.3f}"),
        ('MAPE',                f"{M['MAPE']:.1f}%",            f"{M['cold_sim_MAPE']:.1f}%"),
        ('GEH < 5',             f"{M['pct_GEH_under_5']:.1f}%",  f"{M['cold_pct_GEH_under_5']:.1f}%"),
        ('GEH < 10',            f"{M['pct_GEH_under_10']:.1f}%", f"{M['cold_pct_GEH_under_10']:.1f}%"),
        ('90% coverage',        f"{M['coverage_pct']:.1f}% (target {M['target_pct']:.0f})", '—'),
        ('reliability_score',   f"{M['reliability_seen']:.3f}",  f"{M['reliability_cold']:.3f}"),
        ('rows scored',         f"{M['n_seen_backtest_rows']:,}", f"{M['cold_sim_rows']:,}"),
    ],
    columns=['', 'Seen (LightGBM)', 'Cold-start (GCN)'],
).set_index('')

cold_gain = (1 - M['cold_sim_WAPE'] / M['cold_naive_WAPE']) * 100
print(f"submission rows: {M['submission_rows']:,}  |  invariant self-check: {M['submission_selfcheck']}")
print(f"cold-start beats the naive baseline on WAPE by {cold_gain:.0f}%")
headline

## 1. The daily rhythm of traffic
Everything rests on one fact: traffic is highly repeatable. Weekdays show twin commute peaks; weekends flatten into one midday hump. The climatology layer learns this weekly shape per station.

In [ ]:
display(Image("outputs/figures/fig1_daily_profile.png"))

## 2. Step 1 — LightGBM accuracy on seen series
The LightGBM model reaches MAE 302 on the honest backtest (fit 2024, score H1 2025), just ahead of the seasonal-naive baseline (308). The right panel shows WAPE against the naive→oracle range: 0.392, close to naive and far from the oracle ceiling 0.133. That small gap is a data signal, not a model failure (the +2h offset, left in for the authentic day-one run, caps the achievable score).

In [ ]:
display(Image("outputs/figures/fig2_accuracy_vs_baseline.png"))

## 3. Forecast vs actual — one busy series, one week
A single station-direction over a week. The forecast tracks the daily shape and the weekday/weekend switch; the vertical gaps are the error the metrics summarise.

In [ ]:
display(Image("outputs/figures/fig3_pred_vs_actual.png"))

## 4. Are the uncertainty intervals honest?
The strongest part of the work. The 90% intervals were calibrated with split-conformal residuals and land at **90.0% empirical coverage against a 90% target** — essentially perfect calibration. Wider bands mean the model is honestly less sure.

In [ ]:
display(Image("outputs/figures/fig4_interval_calibration.png"))

## 5. GEH — the traffic-engineering standard (seen series)
DOT practice judges forecasts with GEH. 31% of seen predictions clear GEH < 5 and 54% clear GEH < 10 (mean 11.7). These are modest because 6-month-ahead hourly forecasting is far harder than the microsimulation calibration GEH < 5 was designed for.

In [ ]:
display(Image("outputs/figures/fig5_geh_breakdown.png"))

## 6. Step 2 — GNN cold-start forecasting
Cold-start stations have no history, so a graph convolutional network interpolates their traffic *level* from neighbours in the road network, blended equally with a peer-group median prior; the hourly shape comes from the station's peer group. Evaluated by holding out 15 stations with known history and predicting them as if cold, the GCN reaches **WAPE 0.546 against 0.686** for a naive global-mean baseline, a 20% improvement (exact figures in the table above, read from `metrics.json`).

The holdout is strict: the held-out stations are removed from the peer-group shapes and the level prior as well as from the GCN's loss mask, so a station can never help predict itself through its own peer median. That costs about 0.007 WAPE against the looser version and is the honest number.

Two things that seemed obvious and did **not** work, both measured on this holdout: weighting the graph edges by `edge_type`/`distance_band` (WAPE 0.55 → 9.5, the levels diverge), and widening the node feature set (0.55 → 0.65). Both are documented in `README.md` and `src/gnn.py` so they are not retried blindly. The GCN's backprop is verified against a numerical gradient check (1e-11 relative error) in `src/gnn.py`.

In [ ]:
display(Image("outputs/figures/fig6_cold_start.png"))

## 7. MAPE and GEH to real-world standards (seen vs cold)
Both paths side by side. Seen-series MAPE is 86% and cold-start 117% (cold-start is genuinely harder with no history). The GEH < 5 panel shows the 85% microsimulation-calibration reference line; sitting below it is expected for long-horizon hourly forecasting and is reported honestly for completeness rather than as a pass/fail.

In [ ]:
display(Image("outputs/figures/fig7_realworld_standards.png"))

## Scenario analysis (computed, not hardcoded)
`outputs/scenario_results.csv` holds five what-if scenarios. The official prompts were never in the bundle, so each is a documented what-if *type* (lane closure, event surge, storm, construction, incident reroute) applied to a **real, established station's real forecast**: the volume-change percentage comes from the demand multiplier on that station's forecast, the bounds from the station's own 90% interval width (capped and clipped to physical limits), the recovery time from an exponential-decay model, and the reliability from the station's own score. Swap in real prompts and the same function answers them.

In [ ]:
import pandas as pd; pd.read_csv('outputs/scenario_results.csv')[['scenario_id','station_key','estimated_volume_change_pct','lower_bound_pct','upper_bound_pct','recovery_time_hours','reliability_score']]

## Honest limitations
- **No leakage:** every feature uses only information available at prediction time. Backtest fits 2024 and scores H1 2025; the delivered file fits 2024 + H1 2025 and predicts H2 2025.
- **Cold-start accuracy** is estimated by holding out known stations, since true cold stations have no ground truth on this data. Numbers are honest and clearly worse than the seen path.
- **Reliability and interval widths are derived, not asserted.** `reliability_score` comes from this run's measured accuracy (1 − WAPE per path, with a night-hours haircut). Cold-start bounds come from the holdout's actual/predicted ratio quantiles, rather than treating a mean absolute error as if it were a standard deviation.
- **The submission is self-checked** before it is written: row count against the template, no NaN/inf, `lower_90 <= forecast_volume <= upper_90`, and reliability inside [0, 1].
- **Offset:** the 2024 file's +2h label offset was left in for the authentic day-one run. Correcting it (a later team catch) would sharply improve the seen-series numbers, but that was not part of day one.
- **Scenarios** are computed from real forecasts but use assumed what-if types because official prompts were not supplied.
- Reproduce everything with `python run_pipeline.py`, then the `viz/*.py` scripts.